# Minimum Sample Size Calculation

**Method:** Frequentist z-test for a single proportion (normal approximation to the binomial)

---

$$N = \frac{(z_{\alpha} + z_{\beta})^2 \cdot [\hat{p}(1-\hat{p}) + p_0(1-p_0)]}{(\hat{p} - p_0)^2}$$

---

| Symbol | Meaning |
|--------|---------|
| $N$ | Minimum sample size (visitors) needed |
| $z_{\alpha}$ | z-score for desired confidence level (Type I error) |
| $z_{\beta}$ | z-score for desired statistical power (Type II error) |
| $\hat{p}$ | Expected (true) conversion rate |
| $p_0$ | Target benchmark conversion rate |

### z-score by test type

| Test Type | $z_{\alpha}$ | $\alpha$ | Use when... |
|-----------|-----|----------|-------------|
| **One-sided** | 1.65 | 0.05 (one tail) | You only need to detect CVR **above** target |
| **Two-sided** | 1.96 | 0.025 (each tail) | CVR could be **above or below** target |

### Common power levels

| Power | $z_{\beta}$ | Use when... |
|-------|-------------|-------------|
| 80% | 0.84 | **Standard.** 再テスト可能な場面（A/Bテスト、フェイクドア等）。見逃してもやり直せる |
| 90% | 1.28 | **Conservative.** 再テストが高コスト（時間・予算）、または判断を覆しにくい場面 |
| 95% | 1.65 | **Strict.** 見逃しが重大な損失になる場面（医療試験、大規模投資判断）。80%の約1.5〜2倍の訪問者が必要 |

In [2]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

def calculate_min_sample_size(target_cvr, expected_cvr, two_sided=False, alpha=0.05, power=0.80):
    """
    Calculates the minimum sample size needed to detect a difference
    between expected_cvr and target_cvr with specified confidence and power.

    Statistical method: Frequentist z-test for a single proportion.

    Parameters:
        target_cvr:   The benchmark / minimum viable conversion rate.
        expected_cvr: The conversion rate you believe is the true rate.
        two_sided:    False (default) → one-sided test | True → two-sided test
        alpha:        Significance level (default 0.05 → 95% confidence)
        power:        Statistical power (default 0.80 → 80% chance of detecting a real effect)
    """

    # 1. Validation Logic
    if expected_cvr == target_cvr:
        print(f"❌ ERROR: Expected CVR ({expected_cvr:.1%}) cannot equal Target ({target_cvr:.1%}). There must be a difference to detect.")
        return
    if two_sided is False and expected_cvr <= target_cvr:
        print(f"❌ ERROR: For a one-sided test, Expected CVR ({expected_cvr:.1%}) must be HIGHER than Target ({target_cvr:.1%}).")
        print(f"   ➡ Use two_sided=True if you want to detect a difference in either direction.")
        return

    # 2. Derive z-scores from alpha and power (dynamic, universal)
    z_alpha = norm.ppf(1 - alpha / 2) if two_sided else norm.ppf(1 - alpha)
    z_beta = norm.ppf(power)
    test_label = "Two-Sided" if two_sided else "One-Sided"

    # 3. The Formula: N = (z_α + z_β)² × [p̂(1-p̂) + p₀(1-p₀)] / (p̂ - p₀)²
    z_sum = z_alpha + z_beta
    z_sum_squared = z_sum ** 2
    var_expected = expected_cvr * (1 - expected_cvr)
    var_target = target_cvr * (1 - target_cvr)
    pooled_variance = var_expected + var_target
    effect = expected_cvr - target_cvr
    denominator = effect ** 2

    required_n = int(np.ceil(z_sum_squared * pooled_variance / denominator))

    # 4. Print Results
    print(f"--- SAMPLE SIZE PLANNER ({test_label}) ---")
    print(f"手法   : 頻度論 z検定（正規近似による単一比率の検定）")
    print(f"         正規近似 = 観測データが十分多いとき、二項分布を")
    print(f"         正規分布で近似し、z値で判定する手法。")
    print(f"KPI    : コンバージョン率（CVR）")
    print(f"目標   : 事業成立ライン {target_cvr:.1%}")
    print(f"-" * 55)

    # Alpha derivation — concrete scenario
    if two_sided:
        alpha_input = f"1 - {alpha}/2"
        alpha_ppf_input = 1 - alpha / 2
    else:
        alpha_input = f"1 - {alpha}"
        alpha_ppf_input = 1 - alpha
    print(f"α (有意水準): {alpha} → 信頼度 {1 - alpha:.0%}")
    print(f"z_α = PPF({alpha_input}) = PPF({alpha_ppf_input}) = {z_alpha:.4f}")
    print(f"  α が守るもの = 誤検出の防止（Type I Error / 偽陽性）")
    print(f"  シナリオ: 真のCVRは実は{target_cvr:.1%}（差がない）")
    print(f"  リスク  : たまたまのデータ偏りで「{expected_cvr:.1%}と差がある！」と誤判定")
    print(f"  α の役割: この間違いを犯す確率を {alpha:.0%} 以下に抑える")
    print()

    # Power derivation — concrete scenario
    print(f"Power (検出力): {power:.0%}  （β = {1 - power:.0%}）")
    print(f"z_β = PPF({power}) = {z_beta:.4f}")
    print(f"  Power が守るもの = 見逃しの防止（Type II Error / 偽陰性）")
    print(f"  シナリオ: 真のCVRは本当に{expected_cvr:.1%}（差がある）")
    print(f"  リスク  : データが足りず「{target_cvr:.1%}と差があるとは言えない」と見逃す")
    print(f"  Power の役割: この見逃しを {1 - power:.0%} 以下に抑える（= {power:.0%}で正しく検出）")
    print(f"-" * 55)

    # Formula step-by-step
    print(f"N = (z_α + z_β)² × [p̂(1-p̂) + p₀(1-p₀)] / (p̂ - p₀)²")
    print()
    print(f"  z_α + z_β = {z_alpha:.4f} + {z_beta:.4f} = {z_sum:.4f}")
    print(f"  (z_α + z_β)² = {z_sum:.4f}² = {z_sum_squared:.4f}")
    print()
    print(f"  p̂(1-p̂) = {expected_cvr}×{1-expected_cvr} = {var_expected:.6f}")
    print(f"  p₀(1-p₀) = {target_cvr}×{1-target_cvr} = {var_target:.6f}")
    print(f"  pooled   = {var_expected:.6f} + {var_target:.6f} = {pooled_variance:.6f}")
    print()
    print(f"  (p̂ - p₀)² = ({expected_cvr} - {target_cvr})² = {effect}² = {denominator}")
    print()
    print(f"  N = {z_sum_squared:.4f} × {pooled_variance:.6f} / {denominator}")
    print(f"    = {z_sum_squared * pooled_variance:.4f} / {denominator}")
    print(f"    = {z_sum_squared * pooled_variance / denominator:.1f}")
    print(f"    ≈ {required_n:,} visitors")
    print(f"-" * 55)

    # Interpretation
    print()
    if two_sided:
        print(f"解釈 : 真のCVRが{expected_cvr:.1%}であるとき、{target_cvr:.1%}との差を")
        print(f"       {1-alpha:.0%}の信頼度かつ{power:.0%}の検出力で検出するには")
        print(f"       少なくとも {required_n:,}人 の訪問者が必要。")
        print(f"       （どちらの方向の差も検出する両側検定）")
    else:
        print(f"解釈 : 真のCVRが{expected_cvr:.1%}であるとき、{target_cvr:.1%}を上回ることを")
        print(f"       {1-alpha:.0%}の信頼度かつ{power:.0%}の検出力で証明するには")
        print(f"       少なくとも {required_n:,}人 の訪問者が必要。")
        print(f"       （上方向のみ検出する片側検定）")

    # 5. Generate Data for the 'Planning Curve'
    offset_range = 0.10
    x_values = np.linspace(target_cvr + 0.002, target_cvr + offset_range, 500)
    y_values = (z_sum_squared * (x_values * (1 - x_values) + target_cvr * (1 - target_cvr))) / ((x_values - target_cvr) ** 2)

    # For two-sided, also show the below-target curve
    if two_sided:
        x_below = np.linspace(max(0.001, target_cvr - offset_range), target_cvr - 0.002, 500)
        y_below = (z_sum_squared * (x_below * (1 - x_below) + target_cvr * (1 - target_cvr))) / ((x_below - target_cvr) ** 2)

    # 6. Create Interactive Plotly Graph
    fig = go.Figure()

    primary_blue = '#1f77b4'
    below_orange = '#ff7f0e'

    # A. Above-target curve
    fig.add_trace(go.Scatter(
        x=x_values, y=y_values,
        mode='lines',
        name='Required Traffic (CVR > target)',
        line=dict(color=primary_blue, width=3)
    ))

    # B. Below-target curve (two-sided only)
    if two_sided:
        fig.add_trace(go.Scatter(
            x=x_below, y=y_below,
            mode='lines',
            name='Required Traffic (CVR < target)',
            line=dict(color=below_orange, width=3, dash='dash')
        ))

    # C. Your Specific Point
    fig.add_trace(go.Scatter(
        x=[expected_cvr], y=[required_n],
        mode='markers',
        name='Your Plan',
        marker=dict(color='red', size=12, symbol='diamond')
    ))

    # D. Target reference line
    fig.add_vline(
        x=target_cvr, line_dash="dot", line_color="gray",
        annotation_text=f"Target: {target_cvr:.1%}",
        annotation_position="top left"
    )

    # E. Layout
    fig.update_layout(
        title=dict(
            text=f"<b>Minimum Sample Size Planner ({test_label})</b><br>Target: {target_cvr:.1%}  |  z_α = {z_alpha:.2f}  |  Power = {power:.0%}",
            font=dict(size=20)
        ),
        xaxis_title="If your True Conversion Rate is...",
        yaxis_title="Visitors Needed (N)",
        xaxis=dict(tickformat=".1%", showgrid=True, gridcolor='#eee'),
        yaxis=dict(showgrid=True, gridcolor='#eee'),
        template="plotly_white",
        showlegend=True,
        plot_bgcolor='rgba(0,0,0,0)'
    )

    fig.add_annotation(
        x=expected_cvr, y=required_n,
        text=f"<b>{required_n:,} Visitors</b><br>needed at {expected_cvr:.1%}",
        ax=40, ay=-40,
        font=dict(size=12, color="red"),
        arrowcolor="red"
    )

    fig.show()


# ==========================================
# ENTER YOUR DATA HERE
# ==========================================
# Toggle: two_sided=False (one-sided) | two_sided=True (two-sided)

# TWO-SIDED: "CVR could be above or below 1%, detect either."
calculate_min_sample_size(target_cvr=0.03, expected_cvr=0.005, two_sided=True)

# ONE-SIDED: "I believe CVR is above 1%, prove it."
# calculate_min_sample_size(target_cvr=0.01, expected_cvr=0.02, two_sided=False)

--- SAMPLE SIZE PLANNER (Two-Sided) ---
手法   : 頻度論 z検定（正規近似による単一比率の検定）
         正規近似 = 観測データが十分多いとき、二項分布を
         正規分布で近似し、z値で判定する手法。
KPI    : コンバージョン率（CVR）
目標   : 事業成立ライン 3.0%
-------------------------------------------------------
α (有意水準): 0.05 → 信頼度 95%
z_α = PPF(1 - 0.05/2) = PPF(0.975) = 1.9600
  α が守るもの = 誤検出の防止（Type I Error / 偽陽性）
  シナリオ: 真のCVRは実は3.0%（差がない）
  リスク  : たまたまのデータ偏りで「0.5%と差がある！」と誤判定
  α の役割: この間違いを犯す確率を 5% 以下に抑える

Power (検出力): 80%  （β = 20%）
z_β = PPF(0.8) = 0.8416
  Power が守るもの = 見逃しの防止（Type II Error / 偽陰性）
  シナリオ: 真のCVRは本当に0.5%（差がある）
  リスク  : データが足りず「3.0%と差があるとは言えない」と見逃す
  Power の役割: この見逃しを 20% 以下に抑える（= 80%で正しく検出）
-------------------------------------------------------
N = (z_α + z_β)² × [p̂(1-p̂) + p₀(1-p₀)] / (p̂ - p₀)²

  z_α + z_β = 1.9600 + 0.8416 = 2.8016
  (z_α + z_β)² = 2.8016² = 7.8489

  p̂(1-p̂) = 0.005×0.995 = 0.004975
  p₀(1-p₀) = 0.03×0.97 = 0.029100
  pooled   = 0.004975 + 0.029100 = 0.034075

  (p̂ - p₀)² = (0.005 - 0.03)² = -0.024999999999999998² 